In [58]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver


llm = init_chat_model("gpt-4o-mini")
conn = sqlite3.connect(
    "memory.db",
    check_same_thread = False
    
    )



In [59]:
class State(MessagesState):
    custom_stuff : str

graph_builder = StateGraph(State)


In [60]:
@tool
def get_weather(city: str):
    """Gets weather in city """

    return f"The weather in {city} is sunny."


#bind tools란 ? 
llm_with_tools = llm.bind_tools(tools=[get_weather])

def chatbot(state:State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages":[response]}

    




In [61]:
tool_node = ToolNode(
    tools =[get_weather],
)

In [62]:
graph_builder.add_node("chatbot",chatbot)
graph_builder.add_node("tools",tool_node)

graph_builder.add_edge(START,"chatbot")
graph_builder.add_conditional_edges("chatbot",tools_condition)
graph_builder.add_edge("tools","chatbot")

graph = graph_builder.compile(
    # checkpointer=SqliteSaver(conn),
)
# graph.invoke(
#     {
#         "messages":[
#             {"role":"user","content":"what is the weather in berlin, budapest and bratislava."}
#         ]
#     },
#     config={
#         "configurable": {
#             "thread_id": 1
#         },
#         "recursion_limit":3
#     }
# )



# graph


In [ ]:
# graph.get_state_history(
#     {
#         "configurable":{
#             "thread_id":1
#         }
#     }
# )



# for state in graph.get_state_history(
#     {
#         "configurable":{
#             "thread_id":1
#         }
#     }
# ):
#     print(state.next)



ValueError: No checkpointer set

In [ ]:



async for event in graph.astream(
    {
        "messages":[
            {
                "role":"user",
                "content":"what is the weather in berlin, budapest and bratislava."}
        ]
    },
):
    print(event)
